In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)

In [ ]:
import importlib

In [ ]:
import os
import sys
#root_path = os.path.dirname(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
from env.parameters import P

In [ ]:
from analysis_functions.data_preparation import cohort_type_adjustment

In [ ]:
import dask.dataframe as dd
import pickle
import yaml

In [ ]:
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2


In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
df = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''')

In [ ]:
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
print(df.shape)

In [ ]:
df = cohort_type_adjustment(df, cols_dict)

In [ ]:
from pipeline_functions import *
from custom_plots import *

# Outliers

- Frequencies
  - Pre exac (y1, y2)
  - Pre meds other
  -Pre OCs
  - Pre OCS non-repeat
  - Pre exac_ocs
  - Pre non-exac other diags 
  - Pef
- Environmental variables


In [ ]:
# Function to find upper and lower boundaries
# for normally distributed variables.

def find_normal_boundaries(df, variable):
    """Calculate the boundaries for a Gaussian distribution."""

    upper_boundary = df[variable].mean() + 3 * df[variable].std()
    lower_boundary = df[variable].mean() - 3 * df[variable].std()

    return upper_boundary, lower_boundary




def find_skewed_boundaries(df, variable, distance):
    """Function to find upper and lower boundaries for skewed variables."""
   
    IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)
    lower_boundary = df[variable].quantile(0.25) - (IQR * distance)
    upper_boundary = df[variable].quantile(0.75) + (IQR * distance)

    return upper_boundary, lower_boundary

In [ ]:
def diagnostic_plots(df, variable):
    """Function to create a histogram, a Q-Q plot and a boxplot."""
    plt.figure(figsize=(16, 4))
    plt.subplot(1, 4, 1)
    sns.kdeplot(df[variable])
    plt.title('KDE')
    plt.subplot(1, 4, 2)
    sns.histplot(df[variable], bins=30)
    plt.title('Histogram')
    plt.subplot(1, 4, 3)
    stats.probplot(df[variable].astype(float), dist="norm", plot=plt)
    plt.ylabel('RM quantiles')
    plt.subplot(1, 4, 4)
    sns.boxplot(y=df[variable])
    plt.title('Boxplot')
    plt.show()

In [ ]:
def outlier_analysis(df, col_name):
    """Print out outlier information"""
    print(col_name)
    print("-----------------")
    print(df[col_name].describe())
    print("-----------------")
    print(f'''Max: {np.max(df[col_name])}''')
    print(f'''Min: {np.min(df[col_name])}''')
    print(f'''95th percentile: {df[col_name].quantile(0.95)}''')
    print(f'''5th percentile: {df[col_name].quantile(0.05)}''')
    print(f'''99th percentile: {df[col_name].quantile(0.99)}''')
    print(f'''1st percentile: {df[col_name].quantile(0.01)}''')
    upper_boundary_1, lower_boundary_1 = find_skewed_boundaries(df, col_name, 1.5)
    print(f'''Upper 1.5: {upper_boundary_1}''')
    print(f'''Lower 1.5: {lower_boundary_1}''')
    print(f'''Count upper outlier: {round(df[df[col_name]>upper_boundary_1].shape[0], 2)}''')
    print(f'''Count lower outlier: {round(df[df[col_name]<lower_boundary_1].shape[0],  2)}''')
    print(f'''% upper outlier: %{round(df[df[col_name]>upper_boundary_1].shape[0]/df.shape[0]*100, 2)}''')
    print(f'''% lower outlier: %{round(df[df[col_name]<lower_boundary_1].shape[0]/df.shape[0]*100,  2)}''')

    upper_boundary, lower_boundary = find_skewed_boundaries(df, col_name, 3)
    print(f'''Upper 3: {upper_boundary}''')
    print(f'''Lower 3: {lower_boundary}''')
    print(f'''Count upper outlier: {round(df[df[col_name]>upper_boundary].shape[0], 2)}''')
    print(f'''Count lower outlier: {round(df[df[col_name]<lower_boundary].shape[0],  2)}''')
    print(f'''% upper outlier: %{round(df[df[col_name]>upper_boundary].shape[0]/df.shape[0]*100, 2)}''')
    print(f'''% lower outlier: %{round(df[df[col_name]<lower_boundary].shape[0]/df.shape[0]*100, 2)}''')
    print("\n")

In [ ]:
outlier_analysis(df, "age_cohort_start")


In [ ]:
col = "age_cohort_start"
diagnostic_plots(df, col)

In [ ]:
outlier_analysis(df, "follow_up_asthma_pre_cohort_start")


In [ ]:
col = "follow_up_asthma_pre_cohort_start"
diagnostic_plots(df, col)

In [ ]:
df.shape

In [ ]:
df[df["follow_up_asthma_pre_cohort_start"]>=1].shape

In [ ]:
df[df["follow_up_asthma_pre_cohort_start"]>=2].shape

# pre_exac_1

Exacerbation in a year before the baseline

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_exac_y1"]>=0], "freq_pre_cohort_start_exac_y1")

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_exac_y1"]>0], "freq_pre_cohort_start_exac_y1")


In [ ]:
col = "freq_pre_cohort_start_exac_y1"
diagnostic_plots(df[df[col]>0], col)

# Pre exac y 2

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_exac_y2"]>=0], "freq_pre_cohort_start_exac_y2")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_exac_y2"]>0], "freq_pre_cohort_start_exac_y2")


In [ ]:
col = "freq_pre_cohort_start_exac_y2"
diagnostic_plots(df[df[col]>0], col)

# Non exac diag y1 and y2

Athma diagnosis (non-exacerbation) in year 1 and 2 before the baseline

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y1"]>0], "freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y1")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y2"]>0], 
                 "freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y2")


In [ ]:
col = "freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y1"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
col = "freq_pre_cohort_start_recurrent_asthma_nonexac_diag_y2"
diagnostic_plots(df[df[col]>0], col)

# Medications

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_ocs_y1"]>=0], "freq_pre_cohort_start_meds_ocs_y1")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_ocs_non_repeat_y1"]>=0], "freq_pre_cohort_start_meds_ocs_non_repeat_y1")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_ocs_y2"]>=0], "freq_pre_cohort_start_meds_ocs_y2")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_ocs_non_repeat_y2"]>=0], "freq_pre_cohort_start_meds_ocs_non_repeat_y2")


In [ ]:
col = "freq_pre_cohort_start_meds_ocs_y1"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
col = "freq_pre_cohort_start_meds_ocs_non_repeat_y1"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
col = "freq_pre_cohort_start_meds_ocs_y2"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
col = "freq_pre_cohort_start_meds_ocs_non_repeat_y2"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_other_y1"]>=0], "freq_pre_cohort_start_meds_other_y1")


In [ ]:
outlier_analysis(df[df["freq_pre_cohort_start_meds_other_y2"]>=0], "freq_pre_cohort_start_meds_other_y2")


In [ ]:
col = "freq_pre_cohort_start_meds_other_y1"
diagnostic_plots(df[df[col]>0], col)

In [ ]:
col = "freq_pre_cohort_start_meds_other_y2"
diagnostic_plots(df[df[col]>0], col)

# BMI (We don't need outlier handling. We will binarise this)

In [ ]:
outlier_analysis(df, "bmi_field")


In [ ]:
outlier_analysis(df, "bmi_field_imputed")


In [ ]:
col = "bmi_field"
diagnostic_plots(df, col)

In [ ]:
col = "bmi_field_imputed"
diagnostic_plots(df, col)

# Environment

In [ ]:
outlier_analysis(df, "traffic_intensity_field")


In [ ]:
outlier_analysis(df, "traffic_intensity_field_imputed")


In [ ]:
col = "traffic_intensity_field"
diagnostic_plots(df, col)

In [ ]:
col = "traffic_intensity_field_imputed"
diagnostic_plots(df, col)

In [ ]:
outlier_analysis(df, "distance_major_road_field")


In [ ]:
outlier_analysis(df, "distance_major_road_field_imputed")


In [ ]:
col = "distance_major_road_field_imputed"
diagnostic_plots(df, col)

In [ ]:
outlier_analysis(df, "distance_nearest_road_field")


In [ ]:
outlier_analysis(df, "distance_nearest_road_field_imputed")


In [ ]:
col = "distance_nearest_road_field_imputed"
diagnostic_plots(df, col)

# Categorisation

In [ ]:
def traffic_intensity_quantiles(df: pd.DataFrame, 
                             col: str = "traffic_intensity_field_imputed",
                             new_col: str = "traffic_cat_quintiles") -> pd.DataFrame:
    """Categorisation based on quantiles"""
    df[new_col] = pd.qcut(df[col], q=4, labels=["Low", "Moderate", "High", "Heavy"])
    return df.copy()



In [ ]:
def traffic_intensity_categories(df: pd.DataFrame, 
                                 col: str = "traffic_intensity_field_imputed",
                                 new_col: str = "traffic_cat_custom") -> pd.DataFrame:
    """Custom categorisation"""

    bins = [-float('inf'), 10000, 20000, 30000, float('inf')]
    labels = ["<10k", "[10k-20k)", "[20k-30k)", ">=30k"]
    
    df[new_col] = pd.cut(df[col], bins=bins, labels=labels, right=False)
    return df.copy()

In [ ]:
df = traffic_intensity_quantiles(df)

In [ ]:
sns.countplot(data=df, x="traffic_cat_quintiles")

In [ ]:
df = traffic_intensity_categories(df)

In [ ]:
sns.countplot(data=df, x="traffic_cat_custom")

# Any other missings?

In [ ]:
missing_perc = df.isnull().mean() *100
missing_perc[missing_perc>0.0]

In [ ]:
# Categorical and binary distributions

In [ ]:
df["country_imd"].value_counts(dropna=False)

In [ ]:
df.shape

In [ ]:
df['split'].value_counts()

## Pre post exac

Exacerbation before and after the baseline

In [ ]:
df["flag_post_cohort_start_exac_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_exac_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_exac"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac"].value_counts()/df.shape[0]* 100

## Post exac ocs

Exacerbation with OCS 

In [ ]:
df["flag_post_cohort_start_exac_ocs_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_exac_ocs_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_exac_ocs"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac_ocs_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac_ocs_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_exac_ocs"].value_counts()/df.shape[0]* 100

## Pre Post non-exac diag

Non-exacerbation asthma diagnosis


In [ ]:
df["flag_post_cohort_start_recurrent_asthma_nonexac_diag_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_recurrent_asthma_nonexac_diag_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_recurrent_asthma_nonexac_diag"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_recurrent_asthma_nonexac_diag_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_recurrent_asthma_nonexac_diag_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_recurrent_asthma_nonexac_diag"].value_counts()/df.shape[0]* 100

## OCS

Oral corticosteroid

In [ ]:
df["flag_post_cohort_start_meds_ocs_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_ocs_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_ocs"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs"].value_counts()/df.shape[0]* 100

In [ ]:
# Non repeat
df["flag_post_cohort_start_meds_ocs_non_repeat_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_ocs_non_repeat_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_ocs_non_repeat"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs_non_repeat_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs_non_repeat_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_ocs_non_repeat"].value_counts()/df.shape[0]* 100

# Other medications

In [ ]:
df["flag_post_cohort_start_meds_other_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_other_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_other"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_other_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_other_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_other"].value_counts()/df.shape[0]* 100

In [ ]:
df[df["follow_up_asthma_pre_cohort_start"]>=1]["flag_pre_cohort_start_meds_other_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df[df["follow_up_asthma_pre_cohort_start"]>=2]["flag_pre_cohort_start_meds_other_y2"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_post_cohort_start_meds_all_y1"].value_counts()/df.shape[0]* 100

In [ ]:
df["flag_pre_cohort_start_meds_all_y1"].value_counts()/df.shape[0]* 100

# Other values

In [ ]:
df["cmrbd_bin"].value_counts()/df.shape[0]* 100

In [ ]:
df['sex_female'].value_counts()

In [ ]:
df["age_60+"].value_counts()/df.shape[0]* 100

In [ ]:
df["freq_pre_exac_cat"].value_counts()

In [ ]:
df["freq_pre_meds_all_cat"].value_counts()

In [ ]:
df["cmrbd_bin"].value_counts()

In [ ]:
df["eth_white"].value_counts()

In [ ]:
df['traffic_high_bin_mean'].value_counts()

In [ ]:
df['traffic_high_bin_median'].value_counts()

In [ ]:
df["bmi_30_imputed"].value_counts()

In [ ]:
df["bmi_30_original"].value_counts()